In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# Generating Table

This creates the tables of the accuracies and clustering scores for distinguishing simulated graph distributions. Note that you first have to run `sim_distributions.py`.

In [3]:
# getting the right parameters
model_params = {
        'ER': {'ER_probs': [0.75], 
                'ER_nps': None,
        },
        'RPG':{
            'RPG_probs': [[0.9, 0.1]], 
            'RPG_props': [1/3, 1/3], 
        },
        'SBM': {
            'SBM_probs': [0.8, 0.05], 
            'SBM_blocks': [[1/4, 1/4, 1/4]]
            },
        'RG': {
            'RG_probs': [0.25]
        },
    }
    
models = list(model_params.keys())
all_configs = {}
for model in models:
    model_config = tuple(model_params[model][param] for param in model_params[model].keys())
    all_configs[model] = model_config
configuration = tuple(all_configs[model] for model in models)

# read dataframe
res_df = pd.read_csv(f'./simulations/results/sim_distributions/res_df.csv', header=0, index_col=0)


# group methods
method_group = {}
for method in list(res_df.index):
   if 'toper' in method:
      method_group[method] = 'TopER'
   elif 'kernel' in method:
      method_group[method] = 'Kernel'
   elif 'embedding' in method or 'spectral_zoo' == method:
      method_group[method] = 'Embeddings'
   elif 'divcurves' in method:
      method_group[method] = 'Diversity Curves'
   elif 'div_function' in method or 'one_scale' in method:
      method_group[method] = 'Diversity Profile'
   elif 'persistent_homology' in method:
      method_group[method] = 'Persistent Homology'

      
res_df['group'] = [method_group[method] for method in list(res_df.index)]

#rename rows, only relevant ones, rest will be dropped
new_names = {
      'divcurves_shortest_path_distance': 'Shortest Path',
      'divcurves_pca_shortest_path_distance': 'Shortest Path PCA',
      'div_function_shortest_path_distance': 'Profile Shortest Path', 
      'graph_embedding_netLSD': 'NetLSD',
      'graph_embedding_graph2vec': 'Graph2Vec', 
      'graph_embedding_feathergraph': 'FEATHER',
      'kernel_shortest_path': 'Kernel Shortest Path', 
      'kernel_mds_shortest_path': 'Kernel Shortest Path MDS',
      'kernel_pyramid_match': 'Kernel Pyramid Match', 
      'kernel_mds_pyramid_match': 'Kernel Pyramid Match MDS',
      'kernel_neighborhood_hash': 'Kernel Neighbourhood Hash', 
      'kernel_mds_neighborhood_hash': 'Kernel Neighbourhood Hash MDS',
      'kernel_weisfeiler_lehman': 'Kernel WL',
      'kernel_mds_weisfeiler_lehman': 'Kernel WL MDS', 
      'kernel_core_framework': 'Kernel Core Framework',
      'kernel_mds_core_framework': 'Kernel Core Framework MDS',
      'kernel_weisfeiler_lehman_optimal_assignment': 'Kernel WL Optimal Assignment',
      'kernel_mds_weisfeiler_lehman_optimal_assignment': 'Kernel WL Optimal Assignment MDS', 
      'toper_degree_sub': 'TopER Degree', 
      'toper_popularity_sub': 'TopER Popularity', 
      'toper_forricci_sub': 'TopER Forricci',
      'toper_olricci_sub': 'TopER Olricci',
      'toper_closeness_sub': 'TopER Closeness', 
      'toper_all_sub': 'TopER All Filtrations',
      'spectral_zoo': 'Spectral Zoo', 
      'persistent_homology_degree': 'PH Degree',
      'persistent_homology_ollivier_ricci_curvature': 'PH Olricci',
      'persistent_homology_forman_ricci_curvature': 'PH Fricci',
      'persistent_homology_laplacian': 'PH Laplacian', 
      'persistent_homology_vietoris_rips': 'PH Vietoris Rips',
      'persistent_homology_alpha_complex': 'PH Alpha Complex',
}

df = res_df.rename(new_names)
df = df.loc[list(new_names.values()),:]

# add group to multiindex
df = df.set_index('group', append = True).swaplevel(0,1)
df.index.rename(['Group', 'Method'], inplace = True)

# sort wrt to accuracy, in groups
df = df.groupby(level=0, sort=False).apply(lambda x: x.sort_values('acc_mean', ascending=False)).droplevel(0)

scores = ['acc', 'silh', 'db', 'ch', 'ari_kmeans', 'ari_agglo']
scores_asc = {'acc': False, 'silh': False, 'ari_agglo': False, 'db': True,'ch': False, 'ari_kmeans': False}

res_copy = df.copy()

results_latex = {}

for score in scores:
   mean_col = f'{score}_mean'
   std_col = f'{score}_std'

   if mean_col in res_copy.columns and std_col in res_copy.columns:
      results_latex[score] = res_copy[mean_col].combine(
            res_copy[std_col],
            lambda m, s: '{:.3f}'.format(round(m, 3)) + ' $\pm $ ' + '{:.3f}'.format(round(s, 3))
      )

results_latex_df = pd.DataFrame.from_dict(results_latex)

# Bold the best mean for each score 
for score in scores:
   mean_col = f"{score}_mean"
   display_col =  score
   if mean_col in res_copy.columns:
      if not scores_asc[score]: 
            best_mask = round(res_copy[mean_col],3) == round(res_copy[mean_col].max(),3)
      else:
            best_mask = round(res_copy[mean_col],3) == round(res_copy[mean_col].min(),3)
      results_latex_df.loc[best_mask, display_col] = results_latex_df.loc[best_mask, display_col].apply(
            lambda x: f"\\textbf{{{x}}}"
      )
   else:
      print(f'{mean_col} not in dataframe?')

#final adjustments latex df 
results_latex_df.index.rename([None, 'Method'], inplace = True)
new_cols = {
   'acc': 'Accuracy', 
   'silh': 'Silhouette Score', 
   'db': 'Davies-Bouldin', 
   'ch': 'Calinski-Harabasz', 
   'ari_kmeans': 'ARI k-means', 
   'ari_agglo': 'ARI agglomerative'
}

results_latex_df = results_latex_df.rename(columns=new_cols)
results_latex_df.replace(to_replace='nan $\\pm $ nan', value='-', inplace=True)

print(configuration)
print(results_latex_df.to_latex(multirow=False, escape=False))

(([0.75], None), ([[0.9, 0.1]], [0.3333333333333333, 0.3333333333333333]), ([0.8, 0.05], [[0.25, 0.25, 0.25]]), ([0.25],))
\begin{tabular}{llllllll}
\toprule
                    &            &                     Accuracy &             Silhouette Score &               Davies-Bouldin &               Calinski-Harabasz &                  ARI k-means &            ARI agglomerative \\
{} & Method &                              &                              &                              &                                 &                              &                              \\
\midrule
Diversity Curves & Shortest Path &  \textbf{0.904 $\pm $ 0.042} &           0.427 $\pm $ 0.034 &           0.748 $\pm $ 0.060 &           194.340 $\pm $ 34.081 &  \textbf{0.511 $\pm $ 0.081} &           0.311 $\pm $ 0.119 \\
                    & Shortest Path PCA &           0.888 $\pm $ 0.053 &  \textbf{0.436 $\pm $ 0.034} &  \textbf{0.732 $\pm $ 0.060} &  \textbf{197.355 $\pm $ 35.008} &  \textbf{0

In [ ]:
# getting the right parameters
model_params = {
        'ER': {'ER_probs': [0.75], 
                'ER_nps': None,
        },
        'RPG':{
            'RPG_probs': [[0.9, 0.1]], 
            'RPG_props': [1/3, 1/3], 
        },
        'SBM': {
            'SBM_probs': [0.8, 0.05], 
            'SBM_blocks': [[1/4, 1/4, 1/4]]
            },
        'RG': {
            'RG_probs': [0.25]
        },
    }
    
models = list(model_params.keys())
all_configs = {}
for model in models:
    model_config = tuple(model_params[model][param] for param in model_params[model].keys())
    all_configs[model] = model_config
configuration = tuple(all_configs[model] for model in models)

# read dataframe
res_df = pd.read_csv(f'./simulations/results/sim_distributions/nrange50_100/res_df.csv', header=0, index_col=0)
#step compute cardinality: 5

# group methods
method_group = {}
for method in list(res_df.index):
   if 'toper' in method:
      method_group[method] = 'TopER'
   elif 'kernel' in method:
      method_group[method] = 'Kernel'
   elif 'embedding' in method or 'spectral_zoo' == method:
      method_group[method] = 'Embeddings'
   elif 'divcurves' in method:
      method_group[method] = 'Diversity Curves'
   elif 'div_function' in method or 'one_scale' in method:
      method_group[method] = 'Diversity Profile'
   elif 'persistent_homology' in method:
      method_group[method] = 'Persistent Homology'

      
res_df['group'] = [method_group[method] for method in list(res_df.index)]

#rename rows, only relevant ones, rest will be dropped
new_names = {
      'divcurves_shortest_path_distance': 'Shortest Path',
      'divcurves_pca_shortest_path_distance': 'Shortest Path PCA',
      'div_function_shortest_path_distance': 'Profile Shortest Path', 
      'graph_embedding_netLSD': 'NetLSD',
      'graph_embedding_graph2vec': 'Graph2Vec', 
      'graph_embedding_feathergraph': 'FEATHER',
      'kernel_shortest_path': 'Kernel Shortest Path', 
      'kernel_mds_shortest_path': 'Kernel Shortest Path MDS',
      'kernel_pyramid_match': 'Kernel Pyramid Match', 
      'kernel_mds_pyramid_match': 'Kernel Pyramid Match MDS',
      'kernel_neighborhood_hash': 'Kernel Neighbourhood Hash', 
      'kernel_mds_neighborhood_hash': 'Kernel Neighbourhood Hash MDS',
      'kernel_weisfeiler_lehman': 'Kernel WL',
      'kernel_mds_weisfeiler_lehman': 'Kernel WL MDS', 
      'kernel_core_framework': 'Kernel Core Framework',
      'kernel_mds_core_framework': 'Kernel Core Framework MDS',
      'kernel_weisfeiler_lehman_optimal_assignment': 'Kernel WL Optimal Assignment',
      'kernel_mds_weisfeiler_lehman_optimal_assignment': 'Kernel WL Optimal Assignment MDS', 
      'toper_degree_sub': 'TopER Degree', 
      'toper_popularity_sub': 'TopER Popularity', 
      'toper_forricci_sub': 'TopER Forricci',
      'toper_olricci_sub': 'TopER Olricci',
      'toper_closeness_sub': 'TopER Closeness', 
      'toper_all_sub': 'TopER All Filtrations',
      'spectral_zoo': 'Spectral Zoo', 
      'persistent_homology_degree': 'PH Degree',
      'persistent_homology_ollivier_ricci_curvature': 'PH Olricci',
      'persistent_homology_forman_ricci_curvature': 'PH Fricci',
      'persistent_homology_laplacian': 'PH Laplacian', 
      'persistent_homology_vietoris_rips': 'PH Vietoris Rips',
      'persistent_homology_alpha_complex': 'PH Alpha Complex',
}

df = res_df.rename(new_names)
df = df.loc[list(new_names.values()),:]

# add group to multiindex
df = df.set_index('group', append = True).swaplevel(0,1)
df.index.rename(['Group', 'Method'], inplace = True)

# sort wrt to accuracy, in groups
df = df.groupby(level=0, sort=False).apply(lambda x: x.sort_values('acc_mean', ascending=False)).droplevel(0)

scores = ['acc', 'silh', 'db', 'ch', 'ari_kmeans', 'ari_agglo']
scores_asc = {'acc': False, 'silh': False, 'ari_agglo': False, 'db': True,'ch': False, 'ari_kmeans': False}

res_copy = df.copy()

results_latex = {}

for score in scores:
   mean_col = f'{score}_mean'
   std_col = f'{score}_std'

   if mean_col in res_copy.columns and std_col in res_copy.columns:
      results_latex[score] = res_copy[mean_col].combine(
            res_copy[std_col],
            lambda m, s: '{:.3f}'.format(round(m, 3)) + ' $\pm $ ' + '{:.3f}'.format(round(s, 3))
      )

results_latex_df = pd.DataFrame.from_dict(results_latex)

# Bold the best mean for each score 
for score in scores:
   mean_col = f"{score}_mean"
   display_col =  score
   if mean_col in res_copy.columns:
      if not scores_asc[score]: 
            best_mask = round(res_copy[mean_col],3) == round(res_copy[mean_col].max(),3)
      else:
            best_mask = round(res_copy[mean_col],3) == round(res_copy[mean_col].min(),3)
      results_latex_df.loc[best_mask, display_col] = results_latex_df.loc[best_mask, display_col].apply(
            lambda x: f"\\textbf{{{x}}}"
      )
   else:
      print(f'{mean_col} not in dataframe?')

#final adjustments latex df 
results_latex_df.index.rename([None, 'Method'], inplace = True)
new_cols = {
   'acc': 'Accuracy', 
   'silh': 'Silhouette Score', 
   'db': 'Davies-Bouldin', 
   'ch': 'Calinski-Harabasz', 
   'ari_kmeans': 'ARI k-means', 
   'ari_agglo': 'ARI agglomerative'
}

results_latex_df = results_latex_df.rename(columns=new_cols)
results_latex_df.replace(to_replace='nan $\\pm $ nan', value='-', inplace=True)

print(configuration)
print(results_latex_df.to_latex(multirow=False, escape=False))

(([0.75], None), ([[0.9, 0.1]], [0.3333333333333333, 0.3333333333333333]), ([0.8, 0.05], [[0.25, 0.25, 0.25]]), ([0.25],))
\begin{tabular}{llllllll}
\toprule
                    &                  &                     Accuracy &             Silhouette Score &               Davies-Bouldin &                 Calinski-Harabasz &                  ARI k-means &            ARI agglomerative \\
{} & Method &                              &                              &                              &                                   &                              &                              \\
\midrule
Diversity Curves & Shortest Path &  \textbf{1.000 $\pm $ 0.000} &           0.686 $\pm $ 0.009 &           0.417 $\pm $ 0.013 &           1292.182 $\pm $ 124.201 &           0.755 $\pm $ 0.138 &           0.545 $\pm $ 0.176 \\
                    & Shortest Path PCA &  \textbf{1.000 $\pm $ 0.000} &           0.695 $\pm $ 0.009 &           0.403 $\pm $ 0.013 &           1331.783 $\pm $ 131.40

In [6]:
df

silh_mean    db_mean  \
Group               Method                                                   
Diversity Curves    Shortest Path                      0.685895   0.416718   
                    Shortest Path PCA                  0.695339   0.402883   
Diversity Profile   Profile Shortest Path              0.249913   1.504802   
Embeddings          NetLSD                             0.564212   0.591856   
                    FEATHER                            0.361271   0.869947   
                    Spectral Zoo                       0.300955   0.824624   
                    Graph2Vec                          0.093721   6.406873   
Kernel              Kernel Shortest Path               0.907858        NaN   
                    Kernel Shortest Path MDS           0.847388   0.219886   
                    Kernel WL                          0.794931        NaN   
                    Kernel WL MDS                      0.826739   0.251361   
                    Kernel Core Framework MDS          0.644327   0.495460   
                    Kernel Core Framework              0.622316        NaN   
                    Kernel WL Optimal Assignment       0.032167        NaN   
                    Kernel WL Optimal Assignment MDS   0.095817   2.152891   
                    Kernel Pyramid Match              -0.004319        NaN   
                    Kernel Pyramid Match MDS          -0.032477  34.799833   
                    Kernel Neighbourhood Hash MDS     -0.037285  56.933585   
                    Kernel Neighbourhood Hash         -0.024274        NaN   
TopER               TopER All Filtrations              0.051297   1.963484   
                    TopER Closeness                    0.014251   1.836302   
                    TopER Degree                       0.038695   1.536480   
                    TopER Popularity                   0.005106   3.888972   
                    TopER Olricci                     -0.006826   1.734721   
                    TopER Forricci                     0.048295  16.726045   
Persistent Homology PH Vietoris Rips                   0.277375   1.603607   
                    PH Fricci                          0.127746   1.159742   
                    PH Olricci                         0.135910   1.001598   
                    PH Laplacian                      -0.028093   1.791270   
                    PH Degree                         -0.055423   2.011745   
                    PH Alpha Complex                   0.209755   3.091385   

                                                          ch_mean  silh_std  \
Group               Method                                                    
Diversity Curves    Shortest Path                     1292.182049  0.008867   
                    Shortest Path PCA                 1331.783471  0.008933   
Diversity Profile   Profile Shortest Path              189.983783  0.012069   
Embeddings          NetLSD                             573.590322  0.010561   
                    FEATHER                            511.824271  0.014100   
                    Spectral Zoo                       414.057220  0.016059   
                    Graph2Vec                          175.209092  0.018122   
Kernel              Kernel Shortest Path                      NaN  0.007883   
                    Kernel Shortest Path MDS          6421.048435  0.006249   
                    Kernel WL                                 NaN  0.007702   
                    Kernel WL MDS                     5360.941489  0.006252   
                    Kernel Core Framework MDS         1584.624929  0.012692   
                    Kernel Core Framework                     NaN  0.013915   
                    Kernel WL Optimal Assignment              NaN  0.001650   
                    Kernel WL Optimal Assignment MDS    48.285313  0.017901   
                    Kernel Pyramid Match                      NaN  0.007780   
                    Kernel Pyramid Match MDS             

In [ ]:
# getting the right parameters
model_params = {
        'ER': {'ER_probs': [0.75], 
                'ER_nps': None,
        },
        'RPG':{
            'RPG_probs': [[0.9, 0.1]], 
            'RPG_props': [1/3, 1/3], 
        },
        'SBM': {
            'SBM_probs': [0.8, 0.05], 
            'SBM_blocks': [[1/4, 1/4, 1/4]]
            },
        'RG': {
            'RG_probs': [0.25]
        },
    }
    
models = list(model_params.keys())
all_configs = {}
for model in models:
    model_config = tuple(model_params[model][param] for param in model_params[model].keys())
    all_configs[model] = model_config
configuration = tuple(all_configs[model] for model in models)

# read dataframe
res_df = pd.read_csv(f'./simulations/results/sim_distributions/nrange30_100_stepsize5/res_df.csv', header=0, index_col=0)
#step compute cardinality: 5

# group methods
method_group = {}
for method in list(res_df.index):
   if 'toper' in method:
      method_group[method] = 'TopER'
   elif 'kernel' in method:
      method_group[method] = 'Kernel'
   elif 'embedding' in method or 'spectral_zoo' == method:
      method_group[method] = 'Embeddings'
   elif 'divcurves' in method:
      method_group[method] = 'Diversity Curves'
   elif 'div_function' in method or 'one_scale' in method:
      method_group[method] = 'Diversity Profile'
   elif 'persistent_homology' in method:
      method_group[method] = 'Persistent Homology'

      
res_df['group'] = [method_group[method] for method in list(res_df.index)]

#rename rows, only relevant ones, rest will be dropped
new_names = {
      'divcurves_shortest_path_distance': 'Shortest Path',
      'divcurves_pca_shortest_path_distance': 'Shortest Path PCA',
      'div_function_shortest_path_distance': 'Profile Shortest Path', 
      'graph_embedding_netLSD': 'NetLSD',
      'graph_embedding_graph2vec': 'Graph2Vec', 
      'graph_embedding_feathergraph': 'FEATHER',
      'kernel_shortest_path': 'Kernel Shortest Path', 
      'kernel_mds_shortest_path': 'Kernel Shortest Path MDS',
      'kernel_pyramid_match': 'Kernel Pyramid Match', 
      'kernel_mds_pyramid_match': 'Kernel Pyramid Match MDS',
      'kernel_neighborhood_hash': 'Kernel Neighbourhood Hash', 
      'kernel_mds_neighborhood_hash': 'Kernel Neighbourhood Hash MDS',
      # 'kernel_weisfeiler_lehman': 'Kernel WL', out of memory error!
      # 'kernel_mds_weisfeiler_lehman': 'Kernel WL MDS', 
      'kernel_core_framework': 'Kernel Core Framework',
      'kernel_mds_core_framework': 'Kernel Core Framework MDS',
      'kernel_weisfeiler_lehman_optimal_assignment': 'Kernel WL Optimal Assignment',
      'kernel_mds_weisfeiler_lehman_optimal_assignment': 'Kernel WL Optimal Assignment MDS', 
      'toper_degree_sub': 'TopER Degree', 
      'toper_popularity_sub': 'TopER Popularity', 
      'toper_forricci_sub': 'TopER Forricci',
      'toper_olricci_sub': 'TopER Olricci',
      'toper_closeness_sub': 'TopER Closeness', 
      'toper_all_sub': 'TopER All Filtrations',
      'spectral_zoo': 'Spectral Zoo', 
      'persistent_homology_degree': 'PH Degree',
      'persistent_homology_ollivier_ricci_curvature': 'PH Olricci',
      'persistent_homology_forman_ricci_curvature': 'PH Fricci',
      'persistent_homology_laplacian': 'PH Laplacian', 
      'persistent_homology_vietoris_rips': 'PH Vietoris Rips',
      'persistent_homology_alpha_complex': 'PH Alpha Complex',
}

df = res_df.rename(new_names)
df = df.loc[list(new_names.values()),:]

# add group to multiindex
df = df.set_index('group', append = True).swaplevel(0,1)
df.index.rename(['Group', 'Method'], inplace = True)

# sort wrt to accuracy, in groups
df = df.groupby(level=0, sort=False).apply(lambda x: x.sort_values('acc_mean', ascending=False)).droplevel(0)

scores = ['acc', 'silh', 'db', 'ch', 'ari_kmeans', 'ari_agglo']
scores_asc = {'acc': False, 'silh': False, 'ari_agglo': False, 'db': True,'ch': False, 'ari_kmeans': False}

res_copy = df.copy()

results_latex = {}

for score in scores:
   mean_col = f'{score}_mean'
   std_col = f'{score}_std'

   if mean_col in res_copy.columns and std_col in res_copy.columns:
      results_latex[score] = res_copy[mean_col].combine(
            res_copy[std_col],
            lambda m, s: '{:.3f}'.format(round(m, 3)) + ' $\pm $ ' + '{:.3f}'.format(round(s, 3))
      )

results_latex_df = pd.DataFrame.from_dict(results_latex)

# Bold the best mean for each score 
for score in scores:
   mean_col = f"{score}_mean"
   display_col =  score
   if mean_col in res_copy.columns:
      if not scores_asc[score]: 
            best_mask = round(res_copy[mean_col],3) == round(res_copy[mean_col].max(),3)
      else:
            best_mask = round(res_copy[mean_col],3) == round(res_copy[mean_col].min(),3)
      results_latex_df.loc[best_mask, display_col] = results_latex_df.loc[best_mask, display_col].apply(
            lambda x: f"\\textbf{{{x}}}"
      )
   else:
      print(f'{mean_col} not in dataframe?')

#final adjustments latex df 
results_latex_df.index.rename([None, 'Method'], inplace = True)
new_cols = {
   'acc': 'Accuracy', 
   'silh': 'Silhouette Score', 
   'db': 'Davies-Bouldin', 
   'ch': 'Calinski-Harabasz', 
   'ari_kmeans': 'ARI k-means', 
   'ari_agglo': 'ARI agglomerative'
}

results_latex_df = results_latex_df.rename(columns=new_cols)
results_latex_df.replace(to_replace='nan $\\pm $ nan', value='-', inplace=True)

print(configuration)
print(results_latex_df.to_latex(multirow=False, escape=False))

(([0.75], None), ([[0.9, 0.1]], [0.3333333333333333, 0.3333333333333333]), ([0.8, 0.05], [[0.25, 0.25, 0.25]]), ([0.25],))
\begin{tabular}{llllllll}
\toprule
                    &           &                     Accuracy &             Silhouette Score &               Davies-Bouldin &                 Calinski-Harabasz &                  ARI k-means &            ARI agglomerative \\
{} & Method &                              &                              &                              &                                   &                              &                              \\
\midrule
Diversity Curves & Shortest Path &           0.998 $\pm $ 0.005 &           0.558 $\pm $ 0.012 &           0.591 $\pm $ 0.017 &             848.056 $\pm $ 60.991 &           0.567 $\pm $ 0.033 &           0.328 $\pm $ 0.132 \\
                    & Shortest Path PCA &           0.998 $\pm $ 0.005 &           0.570 $\pm $ 0.012 &           0.569 $\pm $ 0.017 &             870.549 $\pm $ 63.616 &    

In [6]:
df

silh_mean    db_mean  \
Group               Method                                                   
Diversity Curves    Shortest Path                      0.557585   0.590552   
                    Shortest Path PCA                  0.569829   0.568631   
Diversity Profile   Profile Shortest Path              0.092480   2.667957   
Embeddings          NetLSD                             0.310012   1.023193   
                    FEATHER                            0.142306   1.543725   
                    Spectral Zoo                       0.058362   1.380404   
                    Graph2Vec                         -0.067054   4.089604   
Kernel              Kernel Shortest Path               0.768010        NaN   
                    Kernel Core Framework              0.427237        NaN   
                    Kernel Shortest Path MDS           0.738012   0.394266   
                    Kernel Core Framework MDS          0.453802   0.815720   
                    Kernel WL Optimal Assignment       0.011398        NaN   
                    Kernel WL Optimal Assignment MDS   0.002027   5.259829   
                    Kernel Pyramid Match              -0.007579        NaN   
                    Kernel Pyramid Match MDS          -0.029449  24.649097   
                    Kernel Neighbourhood Hash         -0.024748        NaN   
                    Kernel Neighbourhood Hash MDS     -0.033969  62.341556   
TopER               TopER Degree                      -0.076262   2.445262   
                    TopER Closeness                   -0.080211   2.776251   
                    TopER All Filtrations             -0.045225   2.995246   
                    TopER Popularity                  -0.062747   6.469906   
                    TopER Olricci                     -0.087269   2.669990   
                    TopER Forricci                    -0.030448  17.649147   
Persistent Homology PH Vietoris Rips                   0.128168   2.280545   
                    PH Olricci                        -0.034432   1.392882   
                    PH Fricci                         -0.025309   1.551775   
                    PH Alpha Complex                   0.082735   5.259682   
                    PH Laplacian                      -0.135897   2.339386   
                    PH Degree                         -0.144335   2.411096   

                                                          ch_mean  silh_std  \
Group               Method                                                    
Diversity Curves    Shortest Path                      848.055923  0.011829   
                    Shortest Path PCA                  870.548774  0.012005   
Diversity Profile   Profile Shortest Path               87.817997  0.010629   
Embeddings          NetLSD                             199.323505  0.015752   
                    FEATHER                            238.349441  0.013917   
                    Spectral Zoo                       196.225465  0.014286   
                    Graph2Vec                           77.468174  0.014238   
Kernel              Kernel Shortest Path                      NaN  0.018793   
                    Kernel Core Framework                     NaN  0.012969   
                    Kernel Shortest Path MDS          3038.385342  0.013495   
                    Kernel Core Framework MDS          832.899920  0.014266   
                    Kernel WL Optimal Assignment              NaN  0.001368   
                    Kernel WL Optimal Assignment MDS    23.253114  0.008967   
                    Kernel Pyramid Match                      NaN  0.007412   
                    Kernel Pyramid Match MDS             2.301843  0.007798   
                    Kernel Neighbourhood Hash                 NaN  0.005290   
                    Kernel Neighbourhood Hash MDS        0.560784  0.006808   
TopER               TopER Degree                       113.610167  0.012032   
                    TopER Closeness                    

In [ ]:
# getting the right parameters
model_params = {
        'ER': {'ER_probs': [0.75], 
                'ER_nps': None,
        },
        'RPG':{
            'RPG_probs': [[0.9, 0.1]], 
            'RPG_props': [1/3, 1/3], 
        },
        'SBM': {
            'SBM_probs': [0.8, 0.05], 
            'SBM_blocks': [[1/4, 1/4, 1/4]]
            },
        'RG': {
            'RG_probs': [0.25]
        },
    }
    
models = list(model_params.keys())
all_configs = {}
for model in models:
    model_config = tuple(model_params[model][param] for param in model_params[model].keys())
    all_configs[model] = model_config
configuration = tuple(all_configs[model] for model in models)

# read dataframe
res_df = pd.read_csv(f'./simulations/results/sim_distributions/nrange10_100_stepsize5_ngraphs1/res_df.csv', header=0, index_col=0)

# group methods
method_group = {}
for method in list(res_df.index):
   if 'toper' in method:
      method_group[method] = 'TopER'
   elif 'kernel' in method:
      method_group[method] = 'Kernel'
   elif 'embedding' in method or 'spectral_zoo' == method:
      method_group[method] = 'Embeddings'
   elif 'divcurves' in method:
      method_group[method] = 'Diversity Curves'
   elif 'div_function' in method or 'one_scale' in method:
      method_group[method] = 'Diversity Profile'
   elif 'persistent_homology' in method:
      method_group[method] = 'Persistent Homology'

      
res_df['group'] = [method_group[method] for method in list(res_df.index)]

#rename rows, only relevant ones, rest will be dropped
new_names = {
      'divcurves_shortest_path_distance': 'Shortest Path',
      'divcurves_pca_shortest_path_distance': 'Shortest Path PCA',
      'div_function_shortest_path_distance': 'Profile Shortest Path', 
      'graph_embedding_netLSD': 'NetLSD',
      'graph_embedding_graph2vec': 'Graph2Vec', 
      'graph_embedding_feathergraph': 'FEATHER',
      'kernel_shortest_path': 'Kernel Shortest Path', 
      'kernel_mds_shortest_path': 'Kernel Shortest Path MDS',
      'kernel_pyramid_match': 'Kernel Pyramid Match', 
      'kernel_mds_pyramid_match': 'Kernel Pyramid Match MDS',
      'kernel_neighborhood_hash': 'Kernel Neighbourhood Hash', 
      'kernel_mds_neighborhood_hash': 'Kernel Neighbourhood Hash MDS',
      # 'kernel_weisfeiler_lehman': 'Kernel WL', out of memory error!
      # 'kernel_mds_weisfeiler_lehman': 'Kernel WL MDS', 
      'kernel_core_framework': 'Kernel Core Framework',
      'kernel_mds_core_framework': 'Kernel Core Framework MDS',
      'kernel_weisfeiler_lehman_optimal_assignment': 'Kernel WL Optimal Assignment',
      'kernel_mds_weisfeiler_lehman_optimal_assignment': 'Kernel WL Optimal Assignment MDS', 
      'toper_degree_sub': 'TopER Degree', 
      'toper_popularity_sub': 'TopER Popularity', 
      'toper_forricci_sub': 'TopER Forricci',
      'toper_olricci_sub': 'TopER Olricci',
      'toper_closeness_sub': 'TopER Closeness', 
      'toper_all_sub': 'TopER All Filtrations',
      'spectral_zoo': 'Spectral Zoo', 
      'persistent_homology_degree': 'PH Degree',
      'persistent_homology_ollivier_ricci_curvature': 'PH Olricci',
      'persistent_homology_forman_ricci_curvature': 'PH Fricci',
      'persistent_homology_laplacian': 'PH Laplacian', 
      'persistent_homology_vietoris_rips': 'PH Vietoris Rips',
      'persistent_homology_alpha_complex': 'PH Alpha Complex',
}

df = res_df.rename(new_names)
df = df.loc[list(new_names.values()),:]

# add group to multiindex
df = df.set_index('group', append = True).swaplevel(0,1)
df.index.rename(['Group', 'Method'], inplace = True)

# sort wrt to accuracy, in groups
df = df.groupby(level=0, sort=False).apply(lambda x: x.sort_values('acc_mean', ascending=False)).droplevel(0)

scores = ['acc', 'silh', 'db', 'ch', 'ari_kmeans', 'ari_agglo']
scores_asc = {'acc': False, 'silh': False, 'ari_agglo': False, 'db': True,'ch': False, 'ari_kmeans': False}

res_copy = df.copy()

results_latex = {}

for score in scores:
   mean_col = f'{score}_mean'
   std_col = f'{score}_std'

   if mean_col in res_copy.columns and std_col in res_copy.columns:
      results_latex[score] = res_copy[mean_col].combine(
            res_copy[std_col],
            lambda m, s: '{:.3f}'.format(round(m, 3)) + ' $\pm $ ' + '{:.3f}'.format(round(s, 3))
      )

results_latex_df = pd.DataFrame.from_dict(results_latex)

# Bold the best mean for each score 
for score in scores:
   mean_col = f"{score}_mean"
   display_col =  score
   if mean_col in res_copy.columns:
      if not scores_asc[score]: 
            best_mask = round(res_copy[mean_col],3) == round(res_copy[mean_col].max(),3)
      else:
            best_mask = round(res_copy[mean_col],3) == round(res_copy[mean_col].min(),3)
      results_latex_df.loc[best_mask, display_col] = results_latex_df.loc[best_mask, display_col].apply(
            lambda x: f"\\textbf{{{x}}}"
      )
   else:
      print(f'{mean_col} not in dataframe?')

#final adjustments latex df 
results_latex_df.index.rename([None, 'Method'], inplace = True)
new_cols = {
   'acc': 'Accuracy', 
   'silh': 'Silhouette Score', 
   'db': 'Davies-Bouldin', 
   'ch': 'Calinski-Harabasz', 
   'ari_kmeans': 'ARI k-means', 
   'ari_agglo': 'ARI agglomerative'
}

results_latex_df = results_latex_df.rename(columns=new_cols)
results_latex_df.replace(to_replace='nan $\\pm $ nan', value='-', inplace=True)

print(configuration)
print(results_latex_df.to_latex(multirow=False, escape=False))

(([0.75], None), ([[0.9, 0.1]], [0.3333333333333333, 0.3333333333333333]), ([0.8, 0.05], [[0.25, 0.25, 0.25]]), ([0.25],))
\begin{tabular}{llllllll}
\toprule
                    &           &                     Accuracy &             Silhouette Score &               Davies-Bouldin &               Calinski-Harabasz &                  ARI k-means &            ARI agglomerative \\
{} & Method &                              &                              &                              &                                 &                              &                              \\
\midrule
Diversity Curves & Shortest Path &  \textbf{0.989 $\pm $ 0.018} &           0.400 $\pm $ 0.027 &           0.841 $\pm $ 0.057 &           174.464 $\pm $ 20.701 &           0.452 $\pm $ 0.116 &           0.239 $\pm $ 0.019 \\
                    & Shortest Path PCA &           0.981 $\pm $ 0.022 &           0.412 $\pm $ 0.028 &  \textbf{0.815 $\pm $ 0.058} &           177.461 $\pm $ 21.321 &           0

In [3]:
df

silh_mean    db_mean  \
Group               Method                                                   
Diversity Curves    Shortest Path                      0.399547   0.841142   
                    Shortest Path PCA                  0.412274   0.814556   
Diversity Profile   Profile Shortest Path             -0.005707   4.696204   
Embeddings          NetLSD                            -0.080270   2.254459   
                    FEATHER                            0.018777   2.930202   
                    Spectral Zoo                      -0.090816   3.550703   
                    Graph2Vec                         -0.115299   8.947503   
Kernel              Kernel WL Optimal Assignment       0.001226        NaN   
                    Kernel Core Framework              0.244863        NaN   
                    Kernel Shortest Path MDS           0.507928   0.935320   
                    Kernel Shortest Path               0.510584        NaN   
                    Kernel Core Framework MDS          0.231303   1.666085   
                    Kernel WL Optimal Assignment MDS  -0.018208   9.565093   
                    Kernel Pyramid Match              -0.041312        NaN   
                    Kernel Pyramid Match MDS          -0.053850  22.039970   
                    Kernel Neighbourhood Hash         -0.041833        NaN   
                    Kernel Neighbourhood Hash MDS     -0.056516  33.344124   
TopER               TopER All Filtrations             -0.105813   4.436944   
                    TopER Degree                      -0.137318   4.399021   
                    TopER Popularity                  -0.126508   9.172269   
                    TopER Closeness                   -0.131140   4.888090   
                    TopER Forricci                    -0.085365  29.657580   
                    TopER Olricci                     -0.119050   8.963265   
Persistent Homology PH Vietoris Rips                   0.021350   2.847709   
                    PH Olricci                        -0.143917   1.788545   
                    PH Alpha Complex                   0.007277  10.148593   
                    PH Fricci                         -0.122211   1.769898   
                    PH Laplacian                      -0.207238   3.003925   
                    PH Degree                         -0.208338   2.730104   

                                                         ch_mean  silh_std  \
Group               Method                                                   
Diversity Curves    Shortest Path                     174.464398  0.027053   
                    Shortest Path PCA                 177.460755  0.027736   
Diversity Profile   Profile Shortest Path              13.729837  0.016089   
Embeddings          NetLSD                             13.848358  0.048585   
                    FEATHER                            32.490323  0.017559   
                    Spectral Zoo                       17.556630  0.028705   
                    Graph2Vec                          10.059034  0.014090   
Kernel              Kernel WL Optimal Assignment             NaN  0.001824   
                    Kernel Core Framework                    NaN  0.019798   
                    Kernel Shortest Path MDS          180.706528  0.033547   
                    Kernel Shortest Path                     NaN  0.039253   
                    Kernel Core Framework MDS          89.373064  0.023990   
                    Kernel WL Optimal Assignment MDS    6.546438  0.010429   
                    Kernel Pyramid Match                     NaN  0.013114   
                    Kernel Pyramid Match MDS            0.999971  0.013222   
                    Kernel Neighbourhood Hash                NaN  0.009254   
                    Kernel Neighbourhood Hash MDS       0.571578  0.011305   
TopER               TopER All Filtrations              23.680429  0.015913   
                    TopER Degree                       24.777677  0.020004 

only one cardinality, but same number of the classes

In [13]:
# getting the right parameters
model_params = {
        'ER': {'ER_probs': [0.75], 
                'ER_nps': None,
        },
        'RPG':{
            'RPG_probs': [[0.9, 0.1]], 
            'RPG_props': [1/3, 1/3], 
        },
        'SBM': {
            'SBM_probs': [0.8, 0.05], 
            'SBM_blocks': [[1/4, 1/4, 1/4]]
            },
        'RG': {
            'RG_probs': [0.25]
        },
    }
    
models = list(model_params.keys())
all_configs = {}
for model in models:
    model_config = tuple(model_params[model][param] for param in model_params[model].keys())
    all_configs[model] = model_config
configuration = tuple(all_configs[model] for model in models)

# read dataframe
res_df = pd.read_csv(f'./simulations/results/sim_distributions/one_cardinality/res_df.csv', header=0, index_col=0)


# group methods
method_group = {}
for method in list(res_df.index):
   if 'toper' in method:
      method_group[method] = 'TopER'
   elif 'kernel' in method:
      method_group[method] = 'Kernel'
   elif 'embedding' in method or 'spectral_zoo' == method:
      method_group[method] = 'Embeddings'
   elif 'divcurves' in method:
      method_group[method] = 'Diversity Curves'
   elif 'div_function' in method or 'one_scale' in method:
      method_group[method] = 'Diversity Profile'
   elif 'persistent_homology' in method:
      method_group[method] = 'Persistent Homology'

      
res_df['group'] = [method_group[method] for method in list(res_df.index)]

#rename rows, only relevant ones, rest will be dropped
new_names = {
      'divcurves_shortest_path_distance': 'Shortest Path',
      # 'divcurves_pca_shortest_path_distance': 'Shortest Path PCA',
      # 'div_function_shortest_path_distance': 'Profile Shortest Path', 
      'graph_embedding_netLSD': 'NetLSD',
      # 'graph_embedding_graph2vec': 'Graph2Vec', 
      'graph_embedding_feathergraph': 'FEATHER',
      'kernel_shortest_path': 'Kernel Shortest Path', 
      # 'kernel_mds_shortest_path': 'Kernel Shortest Path MDS',
      # 'kernel_pyramid_match': 'Kernel Pyramid Match', 
      # 'kernel_mds_pyramid_match': 'Kernel Pyramid Match MDS',
      # 'kernel_neighborhood_hash': 'Kernel Neighbourhood Hash', 
      # 'kernel_mds_neighborhood_hash': 'Kernel Neighbourhood Hash MDS',
      # 'kernel_weisfeiler_lehman': 'Kernel WL',
      # 'kernel_mds_weisfeiler_lehman': 'Kernel WL MDS', 
      # 'kernel_core_framework': 'Kernel Core Framework',
      # 'kernel_mds_core_framework': 'Kernel Core Framework MDS',
      'kernel_weisfeiler_lehman_optimal_assignment': 'Kernel WL Optimal Assignment',
      # 'kernel_mds_weisfeiler_lehman_optimal_assignment': 'Kernel WL Optimal Assignment MDS', 
      # 'toper_degree_sub': 'TopER Degree', 
      # 'toper_popularity_sub': 'TopER Popularity', 
      # 'toper_forricci_sub': 'TopER Forricci',
      # 'toper_olricci_sub': 'TopER Olricci',
      # 'toper_closeness_sub': 'TopER Closeness', 
      'toper_all_sub': 'TopER All Filtrations',
      'spectral_zoo': 'Spectral Zoo', 
      # 'persistent_homology_degree': 'PH Degree',
      # 'persistent_homology_ollivier_ricci_curvature': 'PH Olricci',
      # 'persistent_homology_forman_ricci_curvature': 'PH Fricci',
      # 'persistent_homology_laplacian': 'PH Laplacian', 
      'persistent_homology_vietoris_rips': 'PH Vietoris Rips',
      # 'persistent_homology_alpha_complex': 'PH Alpha Complex',
}

df = res_df.rename(new_names)
df = df.loc[list(new_names.values()),:]

# add group to multiindex
df = df.set_index('group', append = True).swaplevel(0,1)
df.index.rename(['Group', 'Method'], inplace = True)

# sort wrt to accuracy, in groups
df = df.groupby(level=0, sort=False).apply(lambda x: x.sort_values('acc_mean', ascending=False)).droplevel(0)

scores = [
   'acc', 
   'silh', 
   # 'db', 
   # 'ch', 
   # 'ari_kmeans', 
   # 'ari_agglo'
   ]
scores_asc = {'acc': False, 'silh': False, 'ari_agglo': False, 'db': True,'ch': False, 'ari_kmeans': False}

res_copy = df.copy()

results_latex = {}

for score in scores:
   mean_col = f'{score}_mean'
   std_col = f'{score}_std'

   if mean_col in res_copy.columns and std_col in res_copy.columns:
      results_latex[score] = res_copy[mean_col].combine(
            res_copy[std_col],
            lambda m, s: '{:.3f}'.format(round(m, 3)) + ' $\pm $ ' + '{:.3f}'.format(round(s, 3))
      )

results_latex_df = pd.DataFrame.from_dict(results_latex)

# Bold the best mean for each score 
for score in scores:
   mean_col = f"{score}_mean"
   display_col =  score
   if mean_col in res_copy.columns:
      if not scores_asc[score]: 
            best_mask = round(res_copy[mean_col],3) == round(res_copy[mean_col].max(),3)
      else:
            best_mask = round(res_copy[mean_col],3) == round(res_copy[mean_col].min(),3)
      results_latex_df.loc[best_mask, display_col] = results_latex_df.loc[best_mask, display_col].apply(
            lambda x: f"\\textbf{{{x}}}"
      )
   else:
      print(f'{mean_col} not in dataframe?')

#final adjustments latex df 
results_latex_df.index.rename([None, 'Method'], inplace = True)
new_cols = {
   'acc': 'Accuracy', 
   'silh': 'Silhouette Score', 
   'db': 'Davies-Bouldin', 
   'ch': 'Calinski-Harabasz', 
   'ari_kmeans': 'ARI k-means', 
   'ari_agglo': 'ARI agglomerative'
}

results_latex_df = results_latex_df.rename(columns=new_cols)
results_latex_df.replace(to_replace='nan $\\pm $ nan', value='-', inplace=True)

print(configuration)
print(results_latex_df.to_latex(multirow=False, escape=False))

(([0.75], None), ([[0.9, 0.1]], [0.3333333333333333, 0.3333333333333333]), ([0.8, 0.05], [[0.25, 0.25, 0.25]]), ([0.25],))
\begin{tabular}{llll}
\toprule
                    &                  &                     Accuracy &             Silhouette Score \\
{} & Method &                              &                              \\
\midrule
Diversity Curves & Shortest Path &           0.992 $\pm $ 0.017 &           0.753 $\pm $ 0.019 \\
Embeddings & FEATHER &  \textbf{0.996 $\pm $ 0.012} &           0.688 $\pm $ 0.019 \\
                    & NetLSD &           0.979 $\pm $ 0.021 &           0.648 $\pm $ 0.028 \\
                    & Spectral Zoo &           0.979 $\pm $ 0.021 &           0.675 $\pm $ 0.024 \\
Kernel & Kernel WL Optimal Assignment &  \textbf{0.996 $\pm $ 0.012} &           0.089 $\pm $ 0.003 \\
                    & Kernel Shortest Path &           0.988 $\pm $ 0.019 &  \textbf{0.772 $\pm $ 0.025} \\
TopER & TopER All Filtrations &           0.938 $\pm $ 0.021 &     

In [14]:
print(results_latex_df.to_markdown())

|                                             | Accuracy                    | Silhouette Score            |
|:--------------------------------------------|:----------------------------|:----------------------------|
| ('Diversity Curves', 'Shortest Path')       | 0.992 $\pm $ 0.017          | 0.753 $\pm $ 0.019          |
| ('Embeddings', 'FEATHER')                   | \textbf{0.996 $\pm $ 0.012} | 0.688 $\pm $ 0.019          |
| ('Embeddings', 'NetLSD')                    | 0.979 $\pm $ 0.021          | 0.648 $\pm $ 0.028          |
| ('Embeddings', 'Spectral Zoo')              | 0.979 $\pm $ 0.021          | 0.675 $\pm $ 0.024          |
| ('Kernel', 'Kernel WL Optimal Assignment')  | \textbf{0.996 $\pm $ 0.012} | 0.089 $\pm $ 0.003          |
| ('Kernel', 'Kernel Shortest Path')          | 0.988 $\pm $ 0.019          | \textbf{0.772 $\pm $ 0.025} |
| ('TopER', 'TopER All Filtrations')          | 0.938 $\pm $ 0.021          | 0.398 $\pm $ 0.027          |
| ('Persistent Homology', 'P